# Week 2 Assignment - IMDb Sentiment Analyser
### WnCC Machine Learning Learner Space 2026

---

In this notebook you will build a **complete sentiment analysis pipeline** - from raw text to a trained neural network - entirely from scratch.

**Your pipeline:**
```
Raw Text → Clean → Tokenise → BoW / TF-IDF → Logistic Regression (baseline)
                                            → MLP (neural network)
                                            → Evaluate with F1, Confusion Matrix
```

**Rules:**
- You may use `numpy`, `matplotlib`, `sklearn` for utilities
- For the MLP section, use **PyTorch** (or NumPy if you prefer)
- Do NOT just call `sklearn.Pipeline` and call it done. Implement the concepts.
- Every `# YOUR CODE HERE` block must be filled in

---
**Run the setup cell first. Then work top to bottom.**

In [ ]:
# ========== SETUP ==========
# Run this cell first - it installs dependencies and downloads the dataset

!pip install datasets -q

import numpy as np
import matplotlib.pyplot as plt
import re
import math
from collections import Counter, defaultdict
from datasets import load_dataset

# Load IMDb dataset
print('Loading IMDb dataset...')
dataset = load_dataset('stanfordnlp/imdb', trust_remote_code=True)

# Use a subset for speed
N_TRAIN = 5000
N_TEST  = 1000

# Shuffle and select subsets before extracting lists (we made a similar change in assignment 1)
train_subset = dataset['train'].shuffle(seed=42).select(range(N_TRAIN))
test_subset  = dataset['test'].shuffle(seed=42).select(range(N_TEST))

# Extract texts and labels
train_texts  = train_subset['text']
train_labels = train_subset['label']
test_texts   = test_subset['text']
test_labels  = test_subset['label']

print(f'Loaded {N_TRAIN} training and {N_TEST} test reviews.')
print(f'Sample review: "{train_texts[0][:120]}..."')
print(f'Label: {"Positive" if train_labels[0] == 1 else "Negative"}')

---
## Part 1 - Text Preprocessing

Raw text is messy. Before any model can use it, we need to:
1. **Lowercase** everything (so "Good" and "good" are the same word)
2. **Remove HTML tags** (IMDb reviews often contain `<br />` etc.)
3. **Remove punctuation** (most of it doesn't carry sentiment)
4. **Tokenise** - split into individual words
5. **Remove stopwords** - extremely common words ("the", "is", "a") that add noise

> 💡 Notice: we are NOT removing all punctuation blindly. Negation words like "n't" matter!

In [ ]:
STOPWORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'he', 'him', 'his',
    'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its',
    'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
    'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because',
    'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against',
    'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
    'all', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'than',
    'too', 'very', 's', 't', 'can', 'will', 'just', 'should', "should've", 'now',
    'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn',
    "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn',
    "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn',
    "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn',
    "wouldn't"
])

def preprocess(text):
    """
    Clean and tokenise a raw review string.
    Returns a list of tokens (strings).
    """
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Step 3: Keep only letters and spaces (remove punctuation, numbers)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Step 4: Tokenise by splitting on whitespace
    tokens = text.split()
    # Step 5: Remove stopwords and very short tokens
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

# Preprocess all reviews
print('Preprocessing reviews...')
train_tokens = [preprocess(t) for t in train_texts]
test_tokens  = [preprocess(t) for t in test_texts]

print('Sample tokens:', train_tokens[0][:15])

---
## Part 2 - Bag-of-Words Vectoriser

Build a vocabulary from the training data, then convert each review into a count vector.

**Your task:** Complete the `BagOfWords` class below.

In [ ]:
class BagOfWords:
    def __init__(self, max_vocab=5000):
        self.max_vocab = max_vocab
        self.vocab = {}       # word → index
        self.vocab_size = 0

    def fit(self, token_lists):
        """
        Build vocabulary from training data.
        Keep only the top `max_vocab` most frequent words.
        """
        # Count all word frequencies across all token_lists
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)

        # Take the top max_vocab words
        most_common = counter.most_common(self.max_vocab)
        self.vocab = {word: idx for idx, (word, _) in enumerate(most_common)}
        self.vocab_size = len(self.vocab)
        print(f'Vocabulary built: {self.vocab_size} words')
        return self

    def transform(self, token_lists):
        """
        Convert a list of token lists into a count matrix.
        Output shape: (num_documents, vocab_size)
        """
        # YOUR CODE HERE
        # Hint: for each token_list, create a vector of size vocab_size
        # where index i = count of vocab word i in this document

    def fit_transform(self, token_lists):
        return self.fit(token_lists).transform(token_lists)


bow = BagOfWords(max_vocab=5000)
X_train_bow = bow.fit_transform(train_tokens)
X_test_bow  = bow.transform(test_tokens)
y_train = np.array(train_labels)
y_test  = np.array(test_labels)

print(f'X_train_bow shape: {X_train_bow.shape}')  # (5000, 5000)
print(f'Sparsity: {(X_train_bow == 0).mean():.1%} zeros')

---
## Part 3 - TF-IDF Vectoriser

Improve on raw counts by weighting words by how informative they are.

**Your task:** Complete the `TFIDF` class.

In [ ]:
class TFIDF:
    def __init__(self, max_vocab=5000):
        self.max_vocab = max_vocab
        self.vocab = {}
        self.idf = None

    def fit(self, token_lists):
        # Build vocab (same as BoW)
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)
        most_common = counter.most_common(self.max_vocab)
        self.vocab = {word: idx for idx, (word, _) in enumerate(most_common)}

        # Compute IDF for each vocab word
        # IDF(word) = log((1 + N) / (1 + df(word))) + 1  [smoothed version]
        # where N = number of documents, df = number of docs containing the word
        N = len(token_lists)
        df = np.zeros(len(self.vocab), dtype=np.float32)

        # YOUR CODE HERE
        # Count how many documents each word appears in
        
        self.idf = np.log((1 + N) / (1 + df)) + 1
        print(f'TF-IDF vocab size: {len(self.vocab)}')
        return self

    def transform(self, token_lists):
        # YOUR CODE HERE
        # Step 1: Compute TF matrix (normalised count per document)
        # Step 2: Multiply elementwise by IDF weights
        # Step 3: L2-normalise each row
        

    def fit_transform(self, token_lists):
        return self.fit(token_lists).transform(token_lists)


tfidf = TFIDF(max_vocab=5000)
X_train_tfidf = tfidf.fit_transform(train_tokens)
X_test_tfidf  = tfidf.transform(test_tokens)

print(f'X_train_tfidf shape: {X_train_tfidf.shape}')

---
## Part 4 - Evaluation Toolkit

Implement the metrics from scratch. Then use them everywhere.

In [ ]:
def confusion_matrix(y_true, y_pred):
    """Returns [[TN, FP], [FN, TP]] for binary classification."""
    

def accuracy(y_true, y_pred):
    return # YOUR CODE HERE

def precision(y_true, y_pred):
    # YOUR CODE HERE

def recall(y_true, y_pred):
    # YOUR CODE HERE

def f1_score(y_true, y_pred):
    # YOUR CODE HERE

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Neg (pred)', 'Pos (pred)'])
    ax.set_yticklabels(['Neg (true)', 'Pos (true)'])
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{labels[i][j]}\n{cm[i,j]}',
                    ha='center', va='center', fontsize=13,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_title(title, fontsize=14, pad=12)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

def evaluate(y_true, y_pred, name='Model'):
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy(y_true, y_pred)
    p   = precision(y_true, y_pred)
    r   = recall(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    print(f'\n===== {name} =====')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {p:.4f}')
    print(f'Recall   : {r:.4f}')
    print(f'F1 Score : {f1:.4f}')
    plot_confusion_matrix(cm, title=name)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

print('Evaluation toolkit ready!')

---
## Part 5 - Logistic Regression Baseline

A logistic regression classifier on TF-IDF features. This is your **baseline** - your MLP must beat this.

In [ ]:
from sklearn.linear_model import LogisticRegression

print('Training Logistic Regression baseline...')
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
lr_metrics = evaluate(y_test, y_pred_lr, name='Logistic Regression (TF-IDF Baseline)')

---
## Part 6 - MLP Classifier with PyTorch

Now build a Multi-Layer Perceptron that takes TF-IDF vectors as input and classifies sentiment.

**Architecture:**
```
Input (5000) → Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear(2) → Softmax
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert numpy arrays to PyTorch tensors
X_tr = torch.tensor(X_train_tfidf, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_te = torch.tensor(X_test_tfidf, dtype=torch.float32)
y_te = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_tr, y_tr)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

# ===== Define the MLP =====
class SentimentMLP(nn.Module):
    def __init__(self, input_dim, hidden1=256, hidden2=128, num_classes=2):
        super().__init__()
        self.net = # YOUR CODE HERE

    def forward(self, x):
        return self.net(x)

model     = SentimentMLP(input_dim=5000)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

In [ ]:
# ===== Training Loop =====
EPOCHS = 15
train_losses = []
train_accs   = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(y_batch)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)

    avg_loss = epoch_loss / total
    avg_acc  = correct / total
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)
    print(f'Epoch {epoch+1:02d}/{EPOCHS}  Loss: {avg_loss:.4f}  Train Acc: {avg_acc:.4f}')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, marker='o', color='tomato')
ax1.set_title('Training Loss', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, marker='o', color='steelblue')
ax2.set_title('Training Accuracy', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Evaluate MLP on Test Set =====
model.eval()
with torch.no_grad():
    logits = model(X_te)
    y_pred_mlp = logits.argmax(dim=1).numpy()

mlp_metrics = evaluate(y_test, y_pred_mlp, name='MLP (TF-IDF Input)')

In [ ]:
# ===== Sanity Check =====
print('========== SANITY CHECK ==========')
print(f'Logistic Regression F1: {lr_metrics["f1"]:.4f}')
print(f'MLP F1:                 {mlp_metrics["f1"]:.4f}')

if mlp_metrics['f1'] > 0.70:
    print('✅ PASS: MLP F1 > 0.70')
else:
    print('❌ MLP F1 is below 0.70. Try training for more epochs, tuning hidden sizes, or adjusting learning rate.')

# Quick demo
print('\n===== DEMO =====')
demo_reviews = [
    'This movie was absolutely brilliant. The acting was superb and the story was moving.',
    'Terrible film. Boring, predictable, and a complete waste of time.'
]
for rev in demo_reviews:
    tokens  = preprocess(rev)
    vec     = tfidf.transform([tokens])
    x_demo  = torch.tensor(vec, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        logit  = model(x_demo)
        prob   = torch.softmax(logit, dim=1)
        pred   = prob.argmax().item()
    label = '😊 Positive' if pred == 1 else '😞 Negative'
    conf  = prob[0, pred].item()
    print(f'  Review: "{rev[:60]}..."')
    print(f'  → Prediction: {label}  (confidence: {conf:.2%})\n')

---
## Part 7 - Reflection (Required)

**Fill in this Markdown cell with your answers before submitting.**

1. **What was the F1 score of your Logistic Regression baseline?**  
   _Your answer here_

2. **What was the F1 score of your MLP?**  
   _Your answer here_

3. **Did the MLP outperform the baseline? By how much? Why do you think that is?**  
   _Your answer here_

4. **Look at your confusion matrix. Where does your model make the most mistakes - false positives or false negatives? What kind of reviews do you think it gets wrong?**  
   _Your answer here_

5. **The model is still using TF-IDF as input. What is the fundamental limitation here? What would need to change to build a truly "understanding" model?**  
   _Your answer here (hint: think about what you learned in Topic 3 this week)_

---
**Submit your completed notebook to the WnCC submission form. Good luck! 🚀**